# Module 11 — Classes

You have written classes for four semesters. The syntax will cost you an hour; the
habits are the expensive part: attributes come into existence by assignment rather
than by declaration, there is no `private`, and the Java habit of writing a getter for
every field has no reason to exist here.

## 1. `self` is the first parameter

No `new`, no access modifiers, no field declarations. The receiver is written out —
in the definition, though not at the call.

In [ ]:
class Sensor:
    def __init__(self, tag, unit="C"):  # runs after the object exists, to set it up
        self.tag = tag
        self.unit = unit

    def describe(self):  # `self` is an ordinary first parameter
        return f"{self.tag} in {self.unit}"


sensor = Sensor("TH-04")  # no `new`
print(sensor.describe())  # `self` is passed for you: sensor.describe() is Sensor.describe(sensor)
print(Sensor.describe(sensor))

`__init__` is not a constructor in the Java sense. The object has already been made
when it runs; `__init__` only fills it in, which is why it returns nothing.

`self` is a convention, not a keyword — the parameter could be called anything and
nobody would. What it is not is optional: leave it out and the method cannot reach
the instance at all.

## 2. Attributes are made by assigning to them

There is no list of fields. `self.tag = tag` **creates** the attribute, and the set
of attributes an object has is whatever has been assigned so far.

In [ ]:
class Sensor:
    def __init__(self, tag):
        self.tag = tag


sensor = Sensor("TH-04")
print(sensor.__dict__)  # the attributes, as a dict -- because that is what they are

sensor.calibrated = True  # from outside the class, and it works
print(sensor.__dict__)

other = Sensor("TH-09")
print(hasattr(other, "calibrated"))  # only that one object got it

Which is worth two conclusions.

The **useful** one: an object's state is data, inspectable and changeable, and there
is no ceremony between you and it.

The **costly** one: a misspelled attribute is not an error. `sensor.calibratd = True`
creates a second attribute and the first stays as it was. Nothing raises, nothing
warns, and the bug is a silent no-op — which is the argument for a type checker over
the whole thing, and for `@dataclass(slots=True)` in module 12, which is what actually
turns an unknown attribute into an error. A plain `@dataclass` does not: it accepts
the misspelling like any other class.

## 3. Class attributes, and the one that is shared

A name assigned in the class body belongs to the **class**. Every instance can read
it, and there is one of it.

In [ ]:
class Sensor:
    kind = "thermometer"  # class attribute: one, shared
    seen = []  # also one, shared -- and this one can change

    def __init__(self, tag):
        self.tag = tag  # instance attribute: one per object
        Sensor.seen.append(tag)


a = Sensor("TH-01")
b = Sensor("TH-04")

# Two objects were made. What is in b.seen?
assert b.seen == ...

This is module 04's mutable default argument, one floor up: the list is created once,
when the class body runs, and every instance shares it. It is almost never what
somebody meant to write; what they meant belongs in `__init__`, where each object
gets its own.

Reading and writing are not symmetrical here, and that catches people:

In [ ]:
class Sensor:
    kind = "thermometer"

    def __init__(self, tag):
        self.tag = tag


a = Sensor("TH-01")
b = Sensor("TH-04")

a.kind = "hygrometer"  # does NOT change the class attribute

print(a.kind, b.kind, Sensor.kind)
print(a.__dict__)  # it created an instance attribute that shadows the class one

An attribute lookup checks the instance first and the class second. Assignment writes
to the instance — unless the class attribute is a property with a setter, which is
section 5. So `a.kind = ...` never modifies the class, and `Sensor.seen.append(...)`
above never touched an instance either, because it did not assign at all: it called a
method on the shared object.

**For class attributes: read through the instance, write through the class.** That is
why a mutable class attribute is a trap while an immutable one is a fine way to hold
a constant.

## 4. There is no `private`

Nothing is enforced. What exists is a convention and one piece of mechanism.

- **`_name`** — a single underscore says "not part of the interface, do not rely on
  it". Nothing prevents access. Every tool and every reader honours it anyway.
- **`__name`** — two underscores trigger *name mangling*: inside the class the name
  is rewritten to `_ClassName__name`.

In [ ]:
class Sensor:
    def __init__(self):
        self._internal = 1
        self.__mangled = 2


sensor = Sensor()
print(sensor._internal)  # nothing stops this
print([name for name in vars(sensor)])  # __mangled is not called that any more

try:
    sensor.__mangled
except AttributeError as err:
    print("AttributeError:", err)

print(sensor._Sensor__mangled)  # still reachable, if you insist

Name mangling is **not** a privacy feature, and reading it as one leads people to
sprinkle double underscores everywhere. Its actual purpose is to stop a subclass from
accidentally colliding with an attribute of its parent — the mangled name carries the
defining class in it, so parent and child can each have their own `__cache` without
noticing each other.

Use `_name` for "internal". Use `__name` only when you specifically want that
collision protection, which is rare outside library code.

## 5. `@property`, and why there are no getters here

In Java you write a getter for a field you might one day want to compute, because
changing `sensor.value` to `sensor.getValue()` later means changing every call site.
That pressure does not exist in Python: the call site is `sensor.value` either way,
and `@property` is what turns the second one into a computation.

In [ ]:
class Reading:
    def __init__(self, celsius):
        self.celsius = celsius  # a plain attribute, and it stays one


reading = Reading(21.7)
print(reading.celsius)

In [ ]:
class Reading:
    def __init__(self, celsius):
        self.celsius = celsius

    @property
    def fahrenheit(self):  # read like an attribute, computed on each access
        return self.celsius * 1.8 + 32

    @fahrenheit.setter
    def fahrenheit(self, value):  # assignment goes through here
        self.celsius = (value - 32) / 1.8


reading = Reading(21.7)
print(round(reading.fahrenheit, 2))  # no parentheses -- it is an attribute to the caller

reading.fahrenheit = 212
print(reading.celsius)

So: **start with a plain attribute.** Add a property on the day it has to be
validated, computed, or logged — and no call site changes. The Java-shaped habit of
writing `get_x()` and `set_x()` up front costs you noise now to buy a change you can
make later for free.

`@property` also gives you a read-only attribute, by writing the getter and no setter:
an assignment then raises `AttributeError`.

## 6. `__repr__` and `__str__`

Java has one `toString`. Python has two, for two audiences.

- **`__repr__`** — for you: unambiguous, and by convention it looks like the
  expression that would rebuild the object. This is what the debugger, the traceback,
  and a list of your objects show.
- **`__str__`** — for the user: readable. `print` uses it, and if it is missing,
  `print` falls back to `__repr__`.

Write `__repr__` first, and often only that.

In [ ]:
class Reading:
    def __init__(self, tag, celsius):
        self.tag = tag
        self.celsius = celsius


print(Reading("TH-04", 91.0))  # the default: class name and an address
print([Reading("TH-04", 91.0)])

In [ ]:
class Reading:
    def __init__(self, tag, celsius):
        self.tag = tag
        self.celsius = celsius

    def __repr__(self):
        return f"Reading(tag={self.tag!r}, celsius={self.celsius})"

    def __str__(self):
        return f"{self.tag}: {self.celsius} C"


reading = Reading("TH-04", 91.0)
print(reading)  # __str__
print(repr(reading))  # __repr__
print([reading])  # a container shows the repr of what is in it
print(f"{reading} | {reading!r}")  # !r asks for the repr, as in module 07

A list prints the **repr** of its elements, never the str. That is why an object with
only a `__str__` still shows as `<__main__.Reading object at 0x...>` the moment it is
inside anything.

## 7. `==` on your own class

Module 07 was about `==` comparing content for strings. For a class of your own it
does not, until you say so: the default `==` is identity, which is Java's default too.

In [ ]:
class Reading:
    def __init__(self, celsius):
        self.celsius = celsius


a = Reading(21.7)
b = Reading(21.7)

assert (a == b) == ...
assert (a == a) == ...

In [ ]:
class Reading:
    def __init__(self, celsius):
        self.celsius = celsius

    def __eq__(self, other):
        if not isinstance(other, Reading):
            return NotImplemented  # let the other side try; Python then falls back to identity
        return self.celsius == other.celsius


print(Reading(21.7) == Reading(21.7))
print(Reading(21.7) == "21.7")

And now a consequence that reaches back to module 06. Defining `__eq__` sets
`__hash__` to `None`, so the class becomes **unhashable** — it can no longer be a dict
key or go in a set.

In [ ]:
class Reading:
    def __init__(self, celsius):
        self.celsius = celsius

    def __eq__(self, other):
        return isinstance(other, Reading) and self.celsius == other.celsius


try:
    {Reading(21.7)}
except TypeError as err:
    print("TypeError:", err)

print(Reading.__hash__)

That is the rule from module 06 being enforced: two objects that compare equal must
hash equal, and Python cannot guess how. If you want both, write `__hash__` as well —
over the same fields `__eq__` uses, and only if those fields do not change. Module 12
has `@dataclass`, which does all of this from a declaration.

## 8. Alternative constructors

There is no constructor overloading, for the same reason there is no method
overloading (module 04): `def` binds a name. What Java does with three constructors,
Python does with a `@classmethod` that builds the object and returns it.

In [ ]:
class Reading:
    def __init__(self, celsius):
        self.celsius = celsius

    @classmethod
    def from_fahrenheit(cls, value):  # cls is the class, the way self is the instance
        return cls((value - 32) / 1.8)

    @classmethod
    def from_row(cls, row):
        return cls(float(row["value"]))

    @staticmethod
    def unit():  # no self, no cls: a function that lives here for organisational reasons
        return "C"

    def __repr__(self):
        return f"Reading(celsius={self.celsius:.1f})"


print(Reading(21.7))
print(Reading.from_fahrenheit(212))
print(Reading.from_row({"value": "23.1"}))
print(Reading.unit())

`cls` rather than a hard-coded `Reading` matters the moment somebody subclasses you:
`cls(...)` builds the subclass, `Reading(...)` would not. That is the whole reason
`@classmethod` exists rather than a plain function.

`@staticmethod` takes neither. It is a function in a class's namespace, and the honest
question to ask about one is whether it should be a module-level function instead.
Often it should.

## 9. About the `@`

You have now met it three times — `@property`, `@classmethod`, `@staticmethod` — and
once before, on `@pytest.mark.your_turn` in every test file since module 00.

A line beginning with `@` above a `def` is a **decorator**: it names a function that
is handed your function and gives back something to bind to the name instead. That is
all the syntax is. `@property` returns an object that intercepts attribute access;
`@classmethod` returns one that passes the class instead of the instance.

Module 14 is where you write your own. Until then, reading `@x` as "wrap the function
below in `x`" is enough.

---

`exercises/` is next: `exercise_01.py` to `exercise_06.py`, `exercise_09.py`, and two
in `thinking.md` with nothing to run.

Module 12 takes this further: inheritance, the magic methods that make your type work
with the operators, and `@dataclass`, which writes `__init__`, `__repr__` and `__eq__`
from a list of fields.